# Evaluating Multiple LM Outputs (External)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import \
    check_and_fix_code
from blade_bench.utils import get_dataset_info_path, get_dataset_csv_path

In [3]:
# load files
analysis_subdir_path_1 = "analysis1_output"
analysis_subdir_path_2 = "analysis2_output"
analysis_subdir_path_3 = "analysis3_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"
multirun_filename_3 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)
multirun_path_3 = join(analysis_subdir_path_3, multirun_filename_3)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)
    
with open(multirun_path_3, "r") as file:
    multirun_analyses_3 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']
num_analyses_3 = multirun_analyses_3['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]
analysis_code_filenames_3 = [f"llm_analysis_{i}.py" for i in range(num_analyses_3)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

analysis_code_paths_3 = [join(analysis_subdir_path_3, filename)
                         for filename in analysis_code_filenames_3]

In [4]:
# load dataset info and csv to get task and dataframe
info_path = get_dataset_info_path(multirun_analyses_1["dataset_name"])
data_path = get_dataset_csv_path(multirun_analyses_1["dataset_name"])

with open(info_path, "r") as file:
    info_json = json.load(file)
    
dataset_task = info_json["research_questions"][0]
df = pd.read_csv(data_path)

In [5]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-05 11:38:42.53][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [6]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)
features_3 = format_features(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-05 11:38:43.03][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:38:50.81][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.78 seconds
[2025-12-05 11:38:50.82][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:38:50.84][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:38:57.03][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.19 seconds
[2025-12-05 11:38:57.04][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:38:57.06][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:39:00.81][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  3.75 

In [7]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)
model_info_3 = format_model_info(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-05 11:52:56.28][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:53:08.72][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.44 seconds
[2025-12-05 11:53:08.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:53:08.74][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:53:22.51][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.76 seconds
[2025-12-05 11:53:22.51][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:53:22.52][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:53:44.53][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  22.

In [8]:
conclusions_1 = {}

for i in range(num_analyses_1):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_1[i] = conclusion_str


conclusions_2 = {}

for i in range(num_analyses_2):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_2[i] = conclusion_str
    
conclusions_3 = {}

for i in range(num_analyses_3):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_3[i] = conclusion_str

In [9]:
llm_judge = llm(provider=llm_provider, model=llm_model)
data_head = df.head(10)

[2025-12-05 11:54:54.49][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [10]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final dictionary.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in dictionary format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)


def make_judge_prompt(task, data_head, featA, featB, modelA, modelB, conclA, conclB):
    return (
        f"Research Question / Context:\n{task}\n\n"
        "Here is a sample of the dataset to understand the structure and variables:\n"
        f"{data_head}\n\n"
        "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
        "==================== TRIAL A ====================\n\n"
        "Independent Variables:\n"
        f"{featA['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featA.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featA['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelA}\n\n"
        "Conclusion:\n"
        f"{conclA}\n\n"
        "==================== TRIAL B ====================\n\n"
        "Independent Variables:\n"
        f"{featB['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featB.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featB['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelB}\n\n"
        "Conclusion:\n"
        f"{conclB}\n\n"
        "Now, following your reasoning plan, provide similarity ratings as JSON only."
    )


In [11]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons

In [12]:
### begin with within-group performance
within_group = {1: {}, 2: {}, 3: {}}
for i in range(num_analyses_1):
    for j in range(i + 1, num_analyses_1):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_1[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_1['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_1[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[1][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(i + 1, num_analyses_2):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_2[i],
            features_2[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[2][(i, j)] = response_dict
for i in range(num_analyses_3):
    for j in range(i + 1, num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_3[i],
            features_3[j],
            multirun_analyses_3['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_3[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[3][(i, j)] = response_dict

[2025-12-05 11:54:55.68][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:55:10.40][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.72 seconds
[2025-12-05 11:55:10.41][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:55:10.45][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:55:19.82][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.37 seconds
[2025-12-05 11:55:19.82][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:55:19.86][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:55:30.05][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.2

In [13]:
### now do between-group performance
between_group = { (1, 2): {}, (1, 3): {}, (2, 3): {} }
for i in range(num_analyses_1):
    for j in range(num_analyses_2):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_2[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 2)][(i, j)] = response_dict
for i in range(num_analyses_1):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_3[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 3)][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_2[i],
            features_3[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(2, 3)][(i, j)] = response_dict

[2025-12-05 11:57:05.20][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:57:14.65][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.46 seconds
[2025-12-05 11:57:14.66][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:57:14.71][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:57:25.39][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.68 seconds
[2025-12-05 11:57:25.39][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:57:25.44][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:57:43.75][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  18.3

In [14]:
within_group

{1: {(0, 1): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4},
  (0, 2): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4},
  (1, 2): {'independent_variables': 3,
   'control_variables': 3,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4}},
 2: {(0, 1): {'independent_variables': 4,
   'control_variables': 3,
   'response_variables': 5,
   'model_specification': 3,
   'conclusions': 5,
   'overall_similarity': 4},
  (0, 2): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4},
  (1, 2): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 5,
   'c

In [15]:
within_group[3]

{(0, 1): {'independent_variables': 3,
  'control_variables': 3,
  'response_variables': 3,
  'model_specification': 2,
  'conclusions': 5,
  'overall_similarity': 3},
 (0, 2): {'independent_variables': 4,
  'control_variables': 3,
  'response_variables': 5,
  'model_specification': 3,
  'conclusions': 5,
  'overall_similarity': 4},
 (1, 2): {'independent_variables': 5,
  'control_variables': 4,
  'response_variables': 5,
  'model_specification': 5,
  'conclusions': 4,
  'overall_similarity': 4}}

In [16]:
between_group

{(1,
  2): {(0, 0): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 4,
   'model_specification': 3,
   'conclusions': 5,
   'overall_similarity': 4}, (0, 1): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4}, (0, 2): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 5,
   'conclusions': 5,
   'overall_similarity': 5}, (1, 0): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4}, (1, 1): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4}, (1, 2): {'independent_variables': 4,
   'control_variables': 5,
   'response_variables': 5,
   'model_specification': 4,
   'conclusi

In [24]:
between_group[(1,2)][(1,2)]

{'independent_variables': 4,
 'control_variables': 5,
 'response_variables': 5,
 'model_specification': 4,
 'conclusions': 5,
 'overall_similarity': 5}

In [27]:
features_2[2]

{'independent_variables': [{'description': 'Student-teacher ratio in the district (lower ratio = fewer students per teacher). Calculated as Enrollment / NumTeachers and clipped at the 1st and 99th percentiles to reduce influence of extreme outliers.',
   'columns': ['StudentTeacherRatio_clipped'],
   'transform_code': ["# Compute student-teacher ratio; guard against division by zero\ndf['StudentTeacherRatio'] = df['Enrollment'] / df['NumTeachers']\n\n# Replace infinite values and drop rows missing the core variables\ndf = df.replace([np.inf, -np.inf], np.nan)\ndf = df.dropna(subset=['StudentTeacherRatio', 'AvgScore'])\n\n# Clip extreme StudentTeacherRatio values at 1st and 99th percentiles to reduce influence of outliers\nlower = df['StudentTeacherRatio'].quantile(0.01)\nupper = df['StudentTeacherRatio'].quantile(0.99)\n# If there are not enough non-NA values quantiles might be NaN; guard against that\nif pd.isna(lower) or pd.isna(upper):\n    df['StudentTeacherRatio_clipped'] = df['St

In [26]:
features_1[1]

{'independent_variables': [{'description': 'Student-teacher ratio (primary independent variable). Constructed as number of students divided by number of teachers; log-transformed to reduce skew and used in the regression.',
   'columns': ['log_ST_Ratio'],
   'transform_code': ["df['ST_Ratio'] = df['students'] / df['teachers']\ndf.loc[df['ST_Ratio'] <= 0, 'ST_Ratio'] = np.nan\ndf['log_ST_Ratio'] = np.log(df['ST_Ratio'])"]}],
 'control_variables': [{'description': 'Per-student expenditure (standardized). Controls for resourcing differences across districts. The standardized version (z-score) is used in the model.',
   'is_moderator': False,
   'moderator_on': '',
   'columns': ['expenditure_z'],
   'transform_code': ["z_columns = ['expenditure', 'lunch', 'english', 'income', 'ComputerPerStudent']\nfor col in z_columns:\n    if col in df.columns:\n        col_vals = df[col]\n        mean = col_vals.mean(skipna=True)\n        std = col_vals.std(ddof=0, skipna=True)\n        # If std is zer

In [29]:
model_info_2[2]

'{\n  "model_library": "statsmodels (statsmodels.formula.api - smf)",\n  "model_class": "OLS (Ordinary Least Squares) via smf.ols",\n  "model_parameters": "formula: \'AvgScore ~ StudentTeacherRatio_clipped + [optional controls: NumComputers, ExpenditurePerStudent, DistrictIncomeK, PercEnglishLearners, PercReducedLunch, PercCalWorks] + optional categorical terms C(GradeSpan) + C(County)\'; data: df; fit option: cov_type=\'HC3\' (heteroskedasticity-robust HC3 standard errors)",\n  "model_formula_fitting_code": "rhs_terms = [\'StudentTeacherRatio_clipped\'] + controls + cat_terms\\nformula = \'AvgScore ~ \' + \' + \'.join(rhs_terms)\\n\\n# Fit OLS with robust (HC3) standard errors\\nmodel = smf.ols(formula=formula, data=df)\\nresults = model.fit(cov_type=\'HC3\')"\n}'

In [30]:
model_info_1[1]

'{\n  "model_library": "statsmodels (statsmodels.formula.api / smf)",\n  "model_class": "OLS (ordinary least squares) via smf.ols",\n  "model_parameters": "formula: AvgScore ~ log_ST_Ratio + expenditure_z + lunch_z + english_z + income_z + ComputerPerStudent_z + C(grades) + C(county); categorical factors: C(grades), C(county); fit uses heteroskedasticity-robust standard errors cov_type=\'HC3\'. No other hyperparameters set.",\n  "model_formula_fitting_code": "formula = (\\n    \'AvgScore ~ log_ST_Ratio + expenditure_z + lunch_z + english_z + \'\\n    \'income_z + ComputerPerStudent_z + C(grades) + C(county)\'\\n)\\nresults = smf.ols(formula=formula, data=df).fit(cov_type=\'HC3\')"\n}'

In [17]:
# get average similarity score for each subcategory within each group
average_within_group = {}
for group_id, comparisons in within_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_within_group[group_id] = average_scores

In [18]:
# show average within group rounded to nearest tenth
average_within_group

{1: {'independent_variables': 4.0,
  'control_variables': 3.6666666666666665,
  'response_variables': 5.0,
  'model_specification': 4.0,
  'conclusions': 5.0,
  'overall_similarity': 4.0},
 2: {'independent_variables': 4.333333333333333,
  'control_variables': 3.6666666666666665,
  'response_variables': 5.0,
  'model_specification': 4.0,
  'conclusions': 5.0,
  'overall_similarity': 4.333333333333333},
 3: {'independent_variables': 4.0,
  'control_variables': 3.3333333333333335,
  'response_variables': 4.333333333333333,
  'model_specification': 3.3333333333333335,
  'conclusions': 4.666666666666667,
  'overall_similarity': 3.6666666666666665}}

In [19]:
# get average similarity score for each subcategory between each group
average_between_group = {}
for group_pair, comparisons in between_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_between_group[group_pair] = average_scores

In [20]:
average_between_group

{(1, 2): {'independent_variables': 4.111111111111111,
  'control_variables': 4.111111111111111,
  'response_variables': 4.888888888888889,
  'model_specification': 4.111111111111111,
  'conclusions': 5.0,
  'overall_similarity': 4.333333333333333},
 (1, 3): {'independent_variables': 2.5555555555555554,
  'control_variables': 1.8888888888888888,
  'response_variables': 1.7777777777777777,
  'model_specification': 2.4444444444444446,
  'conclusions': 3.6666666666666665,
  'overall_similarity': 2.0},
 (2, 3): {'independent_variables': 2.6666666666666665,
  'control_variables': 2.3333333333333335,
  'response_variables': 2.6666666666666665,
  'model_specification': 2.6666666666666665,
  'conclusions': 3.888888888888889,
  'overall_similarity': 2.0}}

In [21]:
features_3

{0: {'independent_variables': [{'description': 'District student-teacher ratio (number of students enrolled per full-time-equivalent teacher). Lower values indicate fewer students per teacher.',
    'columns': ['StudentTeacherRatio'],
    'transform_code': ["# Compute student-teacher ratio\ndf['StudentTeacherRatio'] = df['calworks'] / df['teachers']"]}],
  'control_variables': [{'description': 'District expenditure per student (continuous). Controls for district-level funding differences that can affect test scores.',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['expenditure'],
    'transform_code': ["numeric_cols = ['calworks', 'teachers', 'grades', 'rownames', 'expenditure', 'math', 'district', 'english']\nfor col in numeric_cols:\n    if col in df.columns:\n        df[col] = pd.to_numeric(df[col], errors='coerce')"]},
   {'description': 'Percent of students qualifying for reduced-price lunch (socioeconomic status proxy).',
    'is_moderator': False,
    'mode